In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# 1. Load the data
train_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')

# Save ID column for submission
test_ids = test_data['id'] if 'id' in test_data.columns else test_data.index

print(f"Training data shape: {train_data.shape}")
print(f"Testing data shape: {test_data.shape}")

# 2. Exploratory Data Analysis
def explore_data(df, target_col=None):
    print("\n----- Data Overview -----")
    print(f"Shape: {df.shape}")
    print("\n----- Data Types -----")
    print(df.dtypes)
    print("\n----- Missing Values -----")
    missing = df.isnull().sum()
    print(missing[missing > 0])
    
    if target_col and target_col in df.columns:
        print(f"\n----- Target Variable: {target_col} -----")
        print(df[target_col].describe())
        plt.figure(figsize=(10, 6))
        sns.histplot(df[target_col], kde=True)
        plt.title(f"Distribution of {target_col}")
        plt.savefig('target_distribution.png')
        plt.close()

# Run EDA on training data
explore_data(train_data, 'target')

# Combine train and test for preprocessing
target = None
if 'target' in train_data.columns:
    target = train_data['target'].copy()
    train_data = train_data.drop('target', axis=1)

all_data = pd.concat([train_data, test_data], axis=0, sort=False, ignore_index=True)

# 3. Data Preprocessing & Feature Engineering
def preprocess_data(df):
    # Make a copy to avoid modifying the original
    df_processed = df.copy()
    
    # 3.1 Handle date features
    if 'publication_timestamp' in df_processed.columns:
        df_processed['publication_timestamp'] = pd.to_datetime(df_processed['publication_timestamp'])
        df_processed['release_year'] = df_processed['publication_timestamp'].dt.year
        df_processed['release_month'] = df_processed['publication_timestamp'].dt.month
        df_processed['release_day'] = df_processed['publication_timestamp'].dt.day
        df_processed['days_since_2000'] = (df_processed['publication_timestamp'] - 
                                       pd.Timestamp('2000-01-01')).dt.days
        
        # Drop original timestamp column
        df_processed = df_processed.drop('publication_timestamp', axis=1)
    
    # 3.2 Text features
    # Extract length of track and album names
    if 'track_identifier' in df_processed.columns:
        df_processed['track_identifier_length'] = df_processed['track_identifier'].str.len()
    
    # Process creator collective - extract number of artists
    if 'creator_collective' in df_processed.columns:
        df_processed['artist_name_length'] = df_processed['creator_collective'].str.len()
        # If creator_collective has artists separated by some delimiter (assuming comma)
        df_processed['artist_count_v2'] = df_processed['creator_collective'].str.count(',') + 1
    
    # 3.3 Process track features
    track_nums = [0, 1, 2]
    
    # Calculate averages across tracks
    for feature in ['duration_ms', 'rhythmic_cohesion', 'intensity_index', 
                   'organic_texture', 'beat_frequency', 'emotional_charge', 
                   'groove_efficiency', 'organic_immersion']:
        
        # For each feature, calculate average, min, max, range across tracks
        cols = [f"{feature}_{i}" for i in track_nums if f"{feature}_{i}" in df_processed.columns]
        
        if cols:
            df_processed[f'avg_{feature}'] = df_processed[cols].mean(axis=1)
            df_processed[f'min_{feature}'] = df_processed[cols].min(axis=1)
            df_processed[f'max_{feature}'] = df_processed[cols].max(axis=1)
            df_processed[f'range_{feature}'] = df_processed[f'max_{feature}'] - df_processed[f'min_{feature}']
    
    # 3.4 Create interaction features
    # Ratio of duration to beat frequency (track length in beats)
    for i in track_nums:
        dur_col = f'duration_ms_{i}'
        beat_col = f'beat_frequency_{i}'
        if dur_col in df_processed.columns and beat_col in df_processed.columns:
            df_processed[f'beats_per_track_{i}'] = (df_processed[dur_col] / 1000 / 60) * df_processed[beat_col]
    
    # 3.5 Categorical features
    # One-hot encode categorical features
    cat_features = ['weekday_of_release', 'season_of_release', 'lunar_phase']
    for cat in cat_features:
        if cat in df_processed.columns:
            df_processed[cat] = df_processed[cat].astype('category')
    
    # 3.6 Generate more complex features
    # Energy-Danceability product across tracks
    for i in track_nums:
        energy_col = f'intensity_index_{i}'
        dance_col = f'rhythmic_cohesion_{i}'
        if energy_col in df_processed.columns and dance_col in df_processed.columns:
            df_processed[f'energy_dance_product_{i}'] = df_processed[energy_col] * df_processed[dance_col]
    
    # Calculate musical key diversity
    if all(f'harmonic_scale_{i}' in df_processed.columns for i in track_nums):
        df_processed['key_diversity'] = df_processed[[f'harmonic_scale_{i}' for i in track_nums]].nunique(axis=1)
    
    # Calculate mode consistency (are all tracks in same mode?)
    if all(f'tonal_mode_{i}' in df_processed.columns for i in track_nums):
        df_processed['mode_consistency'] = (df_processed[[f'tonal_mode_{i}' for i in track_nums]].nunique(axis=1) == 1).astype(int)
    
    # 3.7 Drop unnecessary columns
    # Now remove original text columns that won't be used for modeling
    text_cols = ['track_identifier', 'creator_collective']
    track_title_cols = [f'composition_label_{i}' for i in track_nums]
    cols_to_drop = text_cols + track_title_cols
    
    return df_processed.drop([col for col in cols_to_drop if col in df_processed.columns], axis=1)

# Preprocess the data
processed_data = preprocess_data(all_data)

# Split back into train and test sets
train_processed = processed_data.iloc[:len(train_data)]
test_processed = processed_data.iloc[len(train_data):]

# 4. Handle Missing Values
# Identify numerical and categorical columns
numerical_cols = train_processed.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = train_processed.select_dtypes(include=['object', 'category']).columns.tolist()

# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numerical_cols),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_cols)
    ])

# 5. Split the data for training and validation
X_train, X_val, y_train, y_val = train_test_split(
    train_processed, target, test_size=0.2, random_state=42
)

# 6. Feature Selection
# Optional: Use feature importance for feature selection
def select_features(X_train, y_train, X_val, threshold=0.001):
    # Train a simple Random Forest to get feature importances
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    
    # Get feature importances
    importances = rf.feature_importances_
    
    # Create DataFrame of features and their importances
    feature_importance = pd.DataFrame({
        'Feature': X_train.columns,
        'Importance': importances
    }).sort_values('Importance', ascending=False)
    
    # Plot feature importances
    plt.figure(figsize=(12, 8))
    sns.barplot(x='Importance', y='Feature', data=feature_importance.head(20))
    plt.title('Top 20 Feature Importances')
    plt.tight_layout()
    plt.savefig('feature_importances.png')
    plt.close()
    
    # Select features above threshold
    selected_features = feature_importance[feature_importance['Importance'] > threshold]['Feature'].tolist()
    print(f"Selected {len(selected_features)} features out of {len(X_train.columns)}")
    
    return selected_features

# Run feature selection
# Note: We'll apply this after preprocessing in the pipeline

# 7. Model Training - Random Forest
# Create the model pipeline
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=200,
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        bootstrap=True,
        random_state=42,
        n_jobs=-1
    ))
])

# Train the model
model_pipeline.fit(X_train, y_train)

# 8. Model Evaluation
def evaluate_model(model, X_val, y_val, model_name='Model'):
    y_pred = model.predict(X_val)
    
    # Calculate metrics
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mae = mean_absolute_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)
    
    print(f"\n----- {model_name} Evaluation -----")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"R² Score: {r2:.4f}")
    
    # Plot predictions vs actual
    plt.figure(figsize=(10, 6))
    plt.scatter(y_val, y_pred, alpha=0.5)
    plt.plot([min(y_val), max(y_val)], [min(y_val), max(y_val)], color='red', linestyle='--')
    plt.title(f"{model_name}: Predicted vs Actual")
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.savefig(f'{model_name.lower().replace(" ", "_")}_predictions.png')
    plt.close()
    
    return rmse, mae, r2, y_pred

# Evaluate the model
rmse, mae, r2, _ = evaluate_model(model_pipeline, X_val, y_val, "Random Forest")

# 9. Hyperparameter Tuning
def tune_model(X_train, y_train, X_val, y_val):
    # Define parameter grid
    param_grid = {
        'model__n_estimators': [100, 200, 300],
        'model__max_depth': [None, 10, 20, 30],
        'model__min_samples_split': [2, 5, 10],
        'model__min_samples_leaf': [1, 2, 4]
    }
    
    # Create a base pipeline
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', RandomForestRegressor(random_state=42, n_jobs=-1))
    ])
    
    # Grid search with cross-validation
    grid_search = GridSearchCV(
        pipeline, param_grid, 
        cv=3, 
        scoring='neg_root_mean_squared_error',
        verbose=1, 
        n_jobs=-1
    )
    
    print("Starting hyperparameter tuning...")
    grid_search.fit(X_train, y_train)
    
    print(f"Best parameters: {grid_search.best_params_}")
    print(f"Best RMSE: {-grid_search.best_score_:.4f}")
    
    # Evaluate the best model
    best_model = grid_search.best_estimator_
    rmse, mae, r2, _ = evaluate_model(best_model, X_val, y_val, "Tuned Random Forest")
    
    return best_model, rmse

# Optional: Run hyperparameter tuning
# Uncomment the below line to run hyperparameter tuning (might take time)
# best_model, best_rmse = tune_model(X_train, y_train, X_val, y_val)

# 10. Feature Importance Analysis
def analyze_feature_importance(model, X_columns):
    # For the RandomForest step in the pipeline
    if hasattr(model[-1], 'feature_importances_'):
        # Get column names after preprocessing
        preprocessor = model[0]
        feature_names = []
        
        # Handle numerical features
        if hasattr(preprocessor, 'transformers_'):
            for name, transformer, column in preprocessor.transformers_:
                if name == 'num':
                    feature_names.extend(column)
                elif name == 'cat':
                    # For categorical features, get one-hot encoded names
                    for cat_col in column:
                        if hasattr(transformer.named_steps['onehot'], 'categories_'):
                            for category in transformer.named_steps['onehot'].categories_:
                                for cat in category:
                                    feature_names.append(f"{cat_col}_{cat}")
        
        # Get feature importances
        importances = model[-1].feature_importances_
        
        # Match lengths (important for debugging)
        if len(importances) != len(feature_names):
            print(f"Warning: Length mismatch - importances:{len(importances)}, features:{len(feature_names)}")
            feature_names = [f"feature_{i}" for i in range(len(importances))]
        
        # Create DataFrame for visualization
        feature_importance = pd.DataFrame({
            'Feature': feature_names[:len(importances)],
            'Importance': importances
        }).sort_values('Importance', ascending=False)
        
        # Plot top features
        plt.figure(figsize=(12, 8))
        sns.barplot(x='Importance', y='Feature', data=feature_importance.head(20))
        plt.title('Top 20 Feature Importances')
        plt.tight_layout()
        plt.savefig('final_feature_importances.png')
        plt.close()
        
        return feature_importance
    
    return None

# Analyze feature importance
importance_df = analyze_feature_importance(model_pipeline, X_train.columns)

# 11. Generate Predictions for Test Set
final_predictions = model_pipeline.predict(test_processed)

# Create submission file
submission = pd.DataFrame({
    'id': test_ids,
    'target': final_predictions
})

# Ensure predictions are within the expected range of 0-100
submission['target'] = submission['target'].clip(0, 100)

# Save submission file
submission.to_csv('submission6.csv', index=False)
print("Submission file created: random_forest_submission.csv")

# 12. Additional Analysis: Learning Curves
def plot_learning_curves(model, X, y, cv=5):
    train_sizes, train_scores, test_scores = learning_curve(
        model, X, y, cv=cv, scoring='neg_root_mean_squared_error',
        train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1
    )
    
    train_rmse = -np.mean(train_scores, axis=1)
    test_rmse = -np.mean(test_scores, axis=1)
    
    plt.figure(figsize=(10, 6))
    plt.plot(train_sizes, train_rmse, 'o-', color='blue', label='Training RMSE')
    plt.plot(train_sizes, test_rmse, 'o-', color='red', label='Validation RMSE')
    plt.xlabel('Training Examples')
    plt.ylabel('RMSE')
    plt.title('Learning Curves')
    plt.legend(loc='best')
    plt.grid(True)
    plt.savefig('learning_curves.png')
    plt.close()

# Optional: Plot learning curves
from sklearn.model_selection import learning_curve
# plot_learning_curves(model_pipeline, train_processed, target)

# 13. Output for Report
print("\n===== FINAL RESULTS =====")
print(f"Random Forest RMSE: {rmse:.4f}")
print(f"Random Forest MAE: {mae:.4f}")
print(f"Random Forest R² Score: {r2:.4f}")
print("\nTop 10 Important Features:")
if importance_df is not None:
    print(importance_df.head(10).to_string(index=False))

print("\nModel trained and evaluated successfully!")

Training data shape: (61609, 62)
Testing data shape: (41074, 61)

----- Data Overview -----
Shape: (61609, 62)

----- Data Types -----
id                          int64
emotional_charge_2        float64
groove_efficiency_1       float64
beat_frequency_1          float64
organic_texture_2         float64
                           ...   
emotional_resonance_0     float64
harmonic_scale_2          float64
intensity_index_2         float64
instrumental_density_0    float64
target                      int64
Length: 62, dtype: object

----- Missing Values -----
emotional_charge_2             2442
groove_efficiency_1             180
beat_frequency_1                386
organic_texture_2               383
composition_label_0            2949
harmonic_scale_1               3305
intensity_index_0              5971
duration_ms_0                  1289
album_name_length              9594
beat_frequency_0               9731
beat_frequency_2               1766
artist_count                   3261
compo